# Information Retrieval & Semantic Search: build it, measure it

A runnable companion to the concept page. Every demo is imported from `information_retrieval.py` (the single source of truth) and **asserts its qualitative point before printing**, so the notebook is self-checking. It runs **offline on CPU** in a few seconds: a real Sentence-BERT is used if reachable, otherwise a deterministic synthetic encoder takes over and every cell still passes.

We build the retrieve-then-rerank engine room piece by piece:
1. lexical **BM25** over an inverted index, and where it goes blind (vocabulary mismatch);
2. **dense** retrieval that crosses that gap by meaning;
3. **RRF** hybrid fusion of the two rankings;
4. a **cross-encoder re-rank** of the cheap retriever's top-k;
5. the metrics — **precision@k, recall@k, MRR, nDCG** — checked against `sklearn`;
6. the **ANN** recall–latency knob, and the offline guarantee.

## Step 0 — load the backend and report the device honestly

`load_encoder()` returns a real `all-MiniLM-L6-v2` when it is reachable, else a deterministic synthetic fallback. Either way the numbers below are reproducible; the BM25 / RRF / metrics / ANN demos are pure numpy and never touch a model.

In [1]:
import numpy as np
import information_retrieval as ir

print(ir.device_report())
encoder = ir.load_encoder()
print(f"backend: {encoder.name}  (real={encoder.is_real})")

device: cpu (detected mps; pinned to CPU for reproducibility)


backend: all-MiniLM-L6-v2  (real=True)


## Step 1 — the inverted index: why lexical search is fast

A forward index maps `doc -> terms`; the **inverted index** flips it to `term -> posting list` of `(doc_id, term_freq)`. To answer a query you touch only the posting lists of the query's terms — never documents that share no word. Note what is **missing**: the query word `car` has *no* posting list, because the gold passage says `automobile`. That hole is the vocabulary gap, made literal.

In [2]:
index = ir.build_inverted_index(ir.CORPUS)
# Contract: 'car' is absent from the index (no document contains it) -> the lexical gap.
assert "car" not in index, "the corpus deliberately never uses the word 'car'"
assert "automobile" in index, "the gold passage uses 'automobile'"

for term in ("automobile", "fix", "engine"):
    print(f"{term:>12} -> {index[term]}")
print(f"{'car':>12} -> (no posting list: the query word is invisible to lexical search)")

  automobile -> [(0, 1)]
         fix -> [(2, 1), (4, 1)]
      engine -> [(0, 1)]
         car -> (no posting list: the query word is invisible to lexical search)


## Step 2 — BM25 from scratch, verified against `rank_bm25`

BM25 = **IDF (rarity) × saturated, length-normalized TF**, summed over query terms. We compute it straight from the inverted index and check the ranking matches the reference `rank_bm25.BM25Okapi` exactly. The gold passage (d0) scores **zero** — it shares no surface word with the query — while two distractors that merely share `fix`/`my` rank above it.

In [3]:
bm = ir.bm25_scores(ir.QUERY, ir.CORPUS)
bm_order = ir.ranking_from_scores(bm)

# Contract: our from-scratch BM25 ranks identically to rank_bm25 (or it is unavailable).
assert ir.bm25_matches_reference(), "from-scratch BM25 must match rank_bm25.BM25Okapi"
# Contract: the gold passage shares no query term, so its BM25 score is exactly 0.
assert bm[ir.GOLD_ID] == 0.0, "gold shares no surface term with the query -> BM25 = 0"

print(f"query: {ir.QUERY!r}\n")
print(f"{'doc':>4} | {'BM25':>6} | text")
for doc_id in bm_order:
    tag = "  <- GOLD" if doc_id == ir.GOLD_ID else ""
    print(f"d{doc_id:<3} | {bm[doc_id]:>6.3f} | {ir.CORPUS[doc_id][:52]}{tag}")
print(f"\nBM25 gold(d0) rank: {ir.rank_of(bm_order, ir.GOLD_ID)}  (matches rank_bm25: verified)")

query: 'How do I fix my car?'

 doc |   BM25 | text
d4   |  4.684 | How to fix my code when the build breaks on my machi
d2   |  3.979 | I had to fix the spelling errors in my report before
d0   |  0.000 | A mechanic repaired the automobile engine and replac  <- GOLD
d1   |  0.000 | A balanced diet and regular exercise keep your heart
d3   |  0.000 | Stock markets fell sharply amid fears of rising inte
d5   |  0.000 | The doctor advised the patient to rest and drink ple
d6   |  0.000 | Photosynthesis converts sunlight into chemical energ
d7   |  0.000 | Quarterly earnings beat analyst expectations, liftin

BM25 gold(d0) rank: 3  (matches rank_bm25: verified)


**Read it:** BM25 puts two off-topic documents (`fix the spelling errors`, `fix my code`) above the passage that actually answers the question, purely on the shared surface words `fix`/`my`. The right answer scores **0**. This is the vocabulary-mismatch wall.

## Step 3 — dense retrieval crosses the vocabulary gap

A bi-encoder maps query and passages into a shared space where **meaning is proximity**. The gold passage's `automobile`/`engine`/`brake` land near the query's `car`, so dense cosine ranks it **#1** — without sharing a single word. We assert only the *qualitative* contract (dense beats lexical on the gold passage), so the cell passes on either backend.

In [4]:
lvd = ir.lexical_vs_dense(encoder)

# Contract: the dense retriever ranks the gold passage strictly higher than BM25 does.
assert lvd["dense_gold_rank"] < lvd["bm25_gold_rank"], "dense must beat lexical on the synonym query"

print(f"{'doc':>4} | {'dense cos':>9} | text")
for doc_id in lvd["dense_order"]:
    tag = "  <- GOLD" if doc_id == ir.GOLD_ID else ""
    print(f"d{doc_id:<3} | {lvd['dense_scores'][doc_id]:>9.3f} | {ir.CORPUS[doc_id][:50]}{tag}")
print(f"\nBM25  gold rank: {lvd['bm25_gold_rank']}")
print(f"Dense gold rank: {lvd['dense_gold_rank']}   (dense crosses the lexical gap)")

 doc | dense cos | text
d0   |     0.393 | A mechanic repaired the automobile engine and repl  <- GOLD
d4   |     0.312 | How to fix my code when the build breaks on my mac
d2   |     0.136 | I had to fix the spelling errors in my report befo
d1   |     0.111 | A balanced diet and regular exercise keep your hea
d3   |     0.051 | Stock markets fell sharply amid fears of rising in
d5   |     0.042 | The doctor advised the patient to rest and drink p
d6   |     0.039 | Photosynthesis converts sunlight into chemical ene
d7   |     0.036 | Quarterly earnings beat analyst expectations, lift

BM25  gold rank: 3
Dense gold rank: 1   (dense crosses the lexical gap)


## Step 4 — hybrid search: fuse the two rankings with RRF

Lexical and dense fail on **opposite** queries, so run both and combine. **Reciprocal Rank Fusion** fuses on *rank position only* — `score(d) = Σ 1/(k + rank)`, `k=60` — so it needs no score calibration and rewards **agreement**. First the page's by-hand example: **A wins despite ranking #1 in neither list**, because it placed high in both.

In [5]:
table = ir.rrf_worked_example()

# Contract: A wins the fusion (ranked 1 & 2 across the two lists) over C's single #1.
assert table[0][0] == "A", "RRF must reward agreement: A wins despite no single #1"

print("BM25 list : [A, B, C, D, E]")
print("dense list: [C, A, F, B, G]\n")
print(f"{'doc':>4} | {'RRF score':>10}")
for doc, score in table:
    print(f"{doc:>4} | {score:>10.5f}")
print("\nfused order:", [d for d, _ in table], "  (A, C, B float up — the agreed docs)")

BM25 list : [A, B, C, D, E]
dense list: [C, A, F, B, G]

 doc |  RRF score
   A |    0.03252
   C |    0.03227
   B |    0.03175
   F |    0.01587
   D |    0.01562
   E |    0.01538
   G |    0.01538

fused order: ['A', 'C', 'B', 'F', 'D', 'E', 'G']   (A, C, B float up — the agreed docs)


In [6]:
# Now fuse the REAL BM25 + dense rankings over the corpus and see the gold passage recover.
hy = ir.hybrid_demo(encoder)

# Contract: hybrid does not rank the gold below BM25's poor rank -- consensus is robust.
assert hy["fused_gold_rank"] <= hy["bm25_gold_rank"], "hybrid should not be worse than BM25 on the gold"

print(f"gold(d0) rank   BM25: {hy['bm25_gold_rank']}   dense: {hy['dense_gold_rank']}   hybrid(RRF): {hy['fused_gold_rank']}")
print("RRF lets the dense signal rescue what BM25 buried — robust to either retriever's miss.")

gold(d0) rank   BM25: 3   dense: 1   hybrid(RRF): 2
RRF lets the dense signal rescue what BM25 buried — robust to either retriever's miss.


## Step 5 — re-ranking: a cross-encoder for the final precision

Stage-1 retrieval is tuned for **recall** (get the right doc *somewhere* in the top-k), not precise order. A **cross-encoder** reads each `(query, passage)` pair **jointly**, so it can judge fine-grained relevance a bi-encoder's single vector cannot. We re-rank BM25's top-k — where the gold passage sits at **#3** — and the cross-encoder lifts it to **#1**. (Offline we use a transparent synthetic cross-encoder; the *shape* — precision at the top — is the lesson.)

In [7]:
rr = ir.rerank_demo(encoder)

# Contract: the cross-encoder re-rank IMPROVES the gold rank, to the very top.
assert rr["gold_rank_after"] == 1, "the cross-encoder must lift the gold passage to #1"
assert rr["gold_rank_after"] < rr["gold_rank_before"], "re-rank must improve the gold rank"

print(f"BM25 candidate order : {[f'd{c}' for c in rr['candidates']]}")
print(f"cross-encoder scores : {[round(float(s), 3) for s in rr['ce_scores']]}")
print(f"re-ranked order      : {[f'd{c}' for c in rr['reranked']]}")
print(f"\ngold(d0) rank: {rr['gold_rank_before']} (BM25) -> {rr['gold_rank_after']} (after re-rank)")

BM25 candidate order : ['d4', 'd2', 'd0', 'd1']
cross-encoder scores : [0.075, 0.075, 1.0, 0.0]
re-ranked order      : ['d0', 'd4', 'd2', 'd1']

gold(d0) rank: 3 (BM25) -> 1 (after re-rank)


## Step 6 — evaluation: precision@k, recall@k, MRR, nDCG (from scratch)

You cannot tune what you cannot measure. We use one graded-relevance list `rel = [3, 2, 3, 0, 1, 2, 0, 0]` (3 = perfect, 0 = irrelevant; 5 relevant of 8). **Precision@k** is the ranker's metric, **recall@k** the retriever's — they trade off as k grows.

In [8]:
rel = np.array(ir.GRADED_RELEVANCE)
n_rel = int((rel > 0).sum())

# Contract: precision@1 = 1.0 (top is relevant); recall@8 = 1.0 (all relevant captured).
assert ir.precision_at_k(rel, 1) == 1.0
assert ir.recall_at_k(rel, len(rel), n_rel) == 1.0

print(f"rel = {list(ir.GRADED_RELEVANCE)}   ({n_rel} relevant of {len(rel)})\n")
print(f"{'k':>2} | {'precision@k':>11} | {'recall@k':>8}")
for k in (1, 3, 5, 8):
    p = ir.precision_at_k(rel, k)
    r = ir.recall_at_k(rel, k, n_rel)
    print(f"{k:>2} | {p:>11.3f} | {r:>8.3f}")

rel = [3, 2, 3, 0, 1, 2, 0, 0]   (5 relevant of 8)

 k | precision@k | recall@k
 1 |       1.000 |    0.200
 3 |       1.000 |    0.600
 5 |       0.800 |    0.800
 8 |       0.625 |    1.000


In [9]:
# MRR cares only WHERE the first relevant result lands -- perfect for QA / 'I'm feeling lucky'.
first_ranks = [1, 3, 2]  # three queries, first-relevant at ranks 1, 3, 2
mrr = ir.mean_reciprocal_rank(first_ranks)

assert abs(mrr - (1 + 1/3 + 1/2) / 3) < 1e-9, "MRR = mean of 1/rank"
print(f"first-relevant ranks {first_ranks} -> MRR = (1/1 + 1/3 + 1/2)/3 = {mrr:.3f}")

first-relevant ranks [1, 3, 2] -> MRR = (1/1 + 1/3 + 1/2)/3 = 0.611


### nDCG, decomposed — and checked against `sklearn`

**nDCG** handles graded relevance *and* position discounting. Build it in three steps: `DCG = Σ relᵢ / log₂(i+1)`; `IDCG` is the DCG of the ideal (sorted) order; `nDCG = DCG/IDCG ∈ [0,1]`. We print the per-rank table, then verify our nDCG equals `sklearn.metrics.ndcg_score`.

> **Gain convention (important):** `sklearn`'s `ndcg_score` uses **linear gain** (`gain = rel`). The other common IR convention is **exponential gain** (`gain = 2^rel − 1`, used by much TREC/learning-to-rank tooling), which rewards highly-relevant hits super-linearly. They give different numbers; our reference check pins the **linear** version to sklearn so the cross-check is exact.

In [10]:
dec = ir.ndcg_decomposition()

print(f"{'rank i':>6} | {'rel':>3} | {'log2(i+1)':>9} | {'rel/log2(i+1)':>13}")
for i, (r, d, c) in enumerate(zip(dec['rel'], dec['discounts'], dec['actual_contrib']), 1):
    print(f"{i:>6} | {int(r):>3} | {d:>9.3f} | {c:>13.3f}")
print(f"\nDCG@8  = {dec['dcg']:.3f}")
print(f"IDCG@8 = {dec['idcg']:.3f}   (ideal order {[int(x) for x in dec['ideal']]})")
print(f"nDCG@8 = DCG/IDCG = {dec['ndcg']:.3f}")

# Contract: our nDCG matches sklearn.metrics.ndcg_score (linear gain) exactly.
assert ir.ndcg_matches_sklearn(), "from-scratch nDCG must match sklearn.metrics.ndcg_score"
print("\nfrom-scratch nDCG == sklearn.metrics.ndcg_score  (verified, linear gain)")

rank i | rel | log2(i+1) | rel/log2(i+1)
     1 |   3 |     1.000 |         3.000
     2 |   2 |     1.585 |         1.262
     3 |   3 |     2.000 |         1.500
     4 |   0 |     2.322 |         0.000
     5 |   1 |     2.585 |         0.387
     6 |   2 |     2.807 |         0.712
     7 |   0 |     3.000 |         0.000
     8 |   0 |     3.170 |         0.000

DCG@8  = 6.861
IDCG@8 = 7.141   (ideal order [3, 3, 2, 2, 1, 0, 0, 0])
nDCG@8 = DCG/IDCG = 0.961

from-scratch nDCG == sklearn.metrics.ndcg_score  (verified, linear gain)


**Read it:** `nDCG@8 = 0.961` — our ranking is 96.1% of the way to the best-possible order for this query. Because it is normalized per query, you can average it across an evaluation set; BEIR, the standard zero-shot retrieval benchmark, reports **nDCG@10**.

## Step 7 — the ANN knob: recall vs cost

Exact nearest-neighbour search is `O(N·d)` per query — perfect recall, ruinous cost. **ANN** trades a little recall for a large speedup. Here an IVF-style retriever (cluster the corpus, search only the `nprobe` nearest cells) sweeps `nprobe`: recall climbs **monotonically** as you probe more cells — and you scan more of the corpus. There is no free lunch, only a chosen point on the recall-vs-cost curve.

In [11]:
sweep = ir.ann_recall_sweep()

# Contract: recall@10 rises monotonically with nprobe (the dial only turns one way).
assert np.all(np.diff(sweep["recall"]) >= -1e-9), "recall must rise monotonically with nprobe"

print(f"IVF over {sweep['n_docs']} vectors, {sweep['n_clusters']} cells; exact baseline {sweep['exact_ms']:.3f} ms/query\n")
print(f"{'nprobe':>6} | {'recall@10':>9} | {'% scanned':>9}")
for nprobe, rec, frac in zip(sweep['nprobes'], sweep['recall'], sweep['fraction_scanned']):
    print(f"{int(nprobe):>6} | {rec:>9.3f} | {frac*100:>8.0f}%")

IVF over 4000 vectors, 16 cells; exact baseline 0.205 ms/query

nprobe | recall@10 | % scanned
     1 |     0.438 |        6%
     2 |     0.593 |       12%
     4 |     0.755 |       25%
     8 |     0.905 |       50%
    16 |     1.000 |      100%


### Optional: the same knob in FAISS

If `faiss` is installed this measures IVF recall against an exact `IndexFlatIP` baseline — the production library doing exactly what the from-scratch sweep above sketched. It is wrapped in a `try/except` so the notebook never fails when faiss is absent.

In [12]:
try:
    import faiss
    np.random.seed(0)
    d, N, nq = 64, 20_000, 200
    xb = np.random.randn(N, d).astype("float32"); faiss.normalize_L2(xb)
    xq = xb[np.random.choice(N, nq, replace=False)].copy()
    flat = faiss.IndexFlatIP(d); flat.add(xb)
    _, gt = flat.search(xq, 10)
    ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, 100, faiss.METRIC_INNER_PRODUCT)
    ivf.train(xb); ivf.add(xb)
    recall = lambda I: float(np.mean([len(set(I[i]) & set(gt[i])) / 10 for i in range(nq)]))
    prev = -1.0
    for nprobe in (1, 8, 32):
        ivf.nprobe = nprobe
        _, I = ivf.search(xq, 10)
        r = recall(I)
        assert r >= prev - 1e-9, "faiss IVF recall must rise with nprobe"
        prev = r
        print(f"FAISS IVF nprobe={nprobe:2d}: recall@10 = {r:.3f}")
except Exception as exc:  # noqa: BLE001
    print(f"(faiss unavailable -- skipped: {type(exc).__name__})")

FAISS IVF nprobe= 1: recall@10 = 0.187
FAISS IVF nprobe= 8: recall@10 = 0.467
FAISS IVF nprobe=32: recall@10 = 0.808


## Step 8 — the offline path always works

To prove the notebook never depends on a network, force the synthetic backend and re-run the headline contract: dense still beats lexical on the vocabulary-mismatch query. Whatever ran above, *this* always passes.

In [13]:
synth = ir.load_encoder(force_synthetic=True)
print("forced backend:", synth.name, "(real=", synth.is_real, ")")
lvd_s = ir.lexical_vs_dense(synth)
assert lvd_s["dense_gold_rank"] < lvd_s["bm25_gold_rank"], "offline dense must still beat lexical"
print(f"offline  BM25 gold rank {lvd_s['bm25_gold_rank']}  ->  dense gold rank {lvd_s['dense_gold_rank']}")
print("the synthetic concept-anchor encoder crosses the lexical gap with no network — always.")

forced backend: synthetic (real= False )
offline  BM25 gold rank 3  ->  dense gold rank 1
the synthetic concept-anchor encoder crosses the lexical gap with no network — always.


## What we saw

- **Lexical (BM25)** is fast and exact-term-strong but blind to synonyms — the gold passage scored **0** because it said `automobile`, not `car`.
- **Dense** retrieval crossed that gap by **meaning**, ranking the gold passage #1.
- **RRF** fused the two on rank alone (no calibration), rewarding **agreement**.
- A **cross-encoder re-rank** lifted the gold passage **3 → 1** in the BM25 candidates — precision exactly where the user looks.
- **nDCG@8 = 0.961**, matching `sklearn` exactly; precision@k falls and recall@k rises with k.
- **ANN** recall climbed monotonically with `nprobe` — the recall-vs-cost dial, measured.

Every number here is produced by `information_retrieval.py`, the same backend the page and the figures use — so the page, the notebook, and the figures cannot disagree.